# 13. GROUP BY · HAVING · AS

*Module: 4 — Advanced querying · Companion summary: `lectures/13-group-by-having-as.md` · SQL engine: SQLite (in-memory)*

## What you will study

- remove duplicates with DISTINCT
- summarize groups of rows with GROUP BY
- filter groups with HAVING — and know why WHERE can't
- rename output columns with AS

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample data — 20 authors, 6 countries

Watch how the same data looks raw → distinct → grouped → filtered:

In [ ]:
q("CREATE TABLE author (author_id TEXT PRIMARY KEY, lastname TEXT, firstname TEXT, email TEXT, country TEXT)")
q("""
INSERT INTO author VALUES
    ('A1','Chong','Raul','RFC@ca.ibm.com','CA'),
    ('A2','Ahuja','Rav','RA@ca.ibm.com','CA'),
    ('A3','Smith','Jack','JS@ca.ibm.com','CA'),
    ('A4','Sharma','Priya','PS@in.ibm.com','IN'),
    ('A5','Patel','Arjun','AP@in.ibm.com','IN'),
    ('A6','Gupta','Neha','NG@in.ibm.com','IN'),
    ('A7','Singh','Rahul',null,'IN'),
    ('A8','Verma','Kavya',null,'IN'),
    ('A9','Reddy','Anil',null,'IN'),
    ('A10','Wang','Wei','WW@cn.ibm.com','CN'),
    ('A11','Li','Ming','LM@cn.ibm.com','CN'),
    ('A12','Zhang','Yu','ZY@cn.ibm.com','CN'),
    ('A13','Liu','Chen',null,'CN'),
    ('A14','Chen','Xiao',null,'CN'),
    ('A15','Zhao','Feng',null,'CN'),
    ('A16','Miller','Dana','DM@us.ibm.com','US'),
    ('A17','Wilson','Kate','KW@us.ibm.com','US'),
    ('A18','Silva','Ana','AS@br.ibm.com','BR'),
    ('A19','Costa','Joao','JC@br.ibm.com','BR'),
    ('A20','Brown','Emma','EB@au.ibm.com','AU')
""")
q("SELECT country FROM author")   # 20 rows, full of duplicates


## 🧪 Step 1 — DISTINCT: collapse the duplicates

In [ ]:
q("SELECT DISTINCT country FROM author")

## 🧪 Step 2 — GROUP BY + COUNT: *how many* per country

In [ ]:
q("SELECT country, COUNT(*) AS \"Count\" FROM author GROUP BY country ORDER BY 2 DESC")

## 🧪 Step 3 — HAVING: filter the GROUPS (not the rows)

In [ ]:
q("SELECT country, COUNT(*) AS \"Count\" FROM author GROUP BY country HAVING COUNT(*) > 4")

## 🧪 Why WHERE can't do this — read the error

In [ ]:
# WHERE runs BEFORE grouping; aggregates aren't computed yet → error:
# SELECT country, COUNT(*) FROM author WHERE COUNT(*) > 4 GROUP BY country

In [ ]:
import sqlite3
try:
    conn.execute("SELECT country, COUNT(*) FROM author WHERE COUNT(*) > 4 GROUP BY country")
except sqlite3.OperationalError as e:
    print(repr(e))   # misuse of aggregate: WHERE filters pre-grouping rows

## 🧪 COUNT(*) vs COUNT(column) — NULLs matter

In [ ]:
q("SELECT COUNT(*) AS all_rows, COUNT(email) AS with_email FROM author")

## 💡 Tips & tricks

- Every column in SELECT must be either in GROUP BY or inside an aggregate.
- HAVING ≈ "a WHERE that runs after grouping" — aggregates allowed inside.
- 📕 Dialect note: quoting an alias with double quotes works everywhere ("Count");
  SQLite is fine with it; SQL Server uses `[]` brackets instead.

## 🎯 Your turn

1. Countries with fewer than 3 authors.
2. Distinct first names, then count authors per first name, largest groups first.

In [ ]:
# 💡 Solution
q("SELECT country, COUNT(*) FROM author GROUP BY country HAVING COUNT(*) < 3")
q("SELECT DISTINCT firstname FROM author")
q("SELECT firstname, COUNT(*) AS n FROM author GROUP BY firstname ORDER BY n DESC")